# Group 5: Checkpoint 4 Submission (Data Loading)

After initial discussions with the client, we learned that all operational data has historically been recorded and stored in various Excel spreadsheets. To successfully migrate this data into our newly designed relational database schema, we propose a structured, multi-step process. 
* First, we will analyze the existing spreadsheets to map the available columns to the appropriate tables and fields in the new schema.
* Second, we will perform necessary data cleaning and transformations — including splitting combined fields, formatting dates, standardizing categorical variables, and handling missing values — to ensure compatibility with the database structure. Ideally we will manipulate the data prior to loading so the final import will have a smooth transition to the new database.
* Finally, we will load the cleaned datasets through csv files, matching the information that is needed to each relation in our proposed schema.

### Step 1: Analyzing existing spreadshits and mapping to new schema

Following our review of the client's information, we identified 4 key spreadsheets that contain historical operational data. Each spreadsheet has multiple columns, which will be mapped, cleaned and migrated to our new schema.
1) ABC_employees.xlsx
   * first_name — Employee’s first name
   * last_name — Employee’s last name
   * department — Department (e.g., Produce, Meat, Bakery)
   * salary — Monthly salary (may require formatting)
   * hours_per_week — Contracted working hours
   * hire_date — Date of hire (some missing values)
   * location_name — Store where the employee works (to be mapped to location_id)
   * shift_date — Date of scheduled shift
   * start_time — Shift start time
   * end_time — Shift end time
   * shift_status — Scheduled, absent, or on leave
2) ABC_products_and_suppliers.xlsx
   * productname — Name of the product
   * category — Product category (e.g., Dairy, Produce, Frozen)
   * unitcost — Cost per unit (historical fluctuations noted)
   * unitprice — Retail price per unit
   * expiration_days — Expected shelf life in days
   * manufacturer_name — Manufacturer's name
   * supplier_name — Supplier’s name
   * supplier_email — Supplier’s contact email
   * supplier_phone — Supplier’s contact number
   * notes — Additional supplier notes (optional)
3) ABC_orders.xlsx
   * order_date — Date the purchase order was placed
   * expected_delivery_date — Expected delivery date
   * delivery_date — Actual delivery date (may be missing for pending orders)
   * supplier_name — Supplier for the order
   * location — Store receiving the order
   * status — Status of the order (varies, may need mapping to ('Ordered', 'Received', 'Cancelled'))
   * productname — Product ordered
   * units — Units ordered
   * unit_cost — Cost per unit at the time of order
4) ABC_sales_and_customers.xlsx
   * sale_date — Date of sale
   * location_name — Store location of the sale
   * customer_first_name — Customer first name
   * customer_last_name — Customer last name
   * age — Customer age
   * gender — Customer gender
   * email — Customer email (sometimes missing)
   * loyalty_member — Whether the customer is a loyalty member (yes/no)
   * product_name — Product sold
   * quantity — Quantity sold
   * unit_price — Price at time of sale
   * discount_applied — Discount applied during sale
   * promotion_name — Name of promotion (if any)
   * notes - Notes (including 'returns')
5) ABC_accounting.xlsx
   * expense_id	- Unique identifier for each expense
   * category - The type of expense
   * amount	- The dollar amount spent for that particular expense.
   * expense_date - The date when the expense was incurred or recorded.
   * location_name - Name the store or location associated with the expense
   * supplier_id - The ID of the supplier related to the expense, if applicable
   * description - Additional notes or a short explanation about the expense

Based from this information, we can map the data inputs from identified columns directly to our schema:
* Employees: employees and staffing
* Products and suppliers: products, manufacturers and suppliers
* Orders: purchase_orders, deliveries and purchase_order_details
* Sales and customers: customers, sales_orders, sale_order_details and promotions

Other tables such as inventory, transactions, expenses and returns will not be manually populated from the client spreadsheets. Instead, they will be populated through an automated process and triggers. For example:
* purchase_order_details and sale_order_details will be populated based on items from the order data.
* inventory will automatically be updated by triggers when deliveries are finalized or when products are sold.

### [Back-Up] Data generation

Since we do not have data from the client, we wrote code for an automated, randomized data generation of the 4 excel files. We used some of the following assumptions for the data:
* Departments, suppliers and products randomized in different tables will come from the same initial list (so there aren't 'new' products that are in sales but not sold by suppliers).
* Added 1/4 of customer profiles from sales orders (as not everyone will give personal information and become a member)
* Added 3 'outliers' or 'seasonal' product price spikes: eggs, avocados, and ice cream (sold more in summer than winter)

In [29]:
# Employees table (100 records)
import pandas as pd
import random
from faker import Faker

faker = Faker()

# Configurable options
departments = ['Produce', 'Meat', 'Bakery', 'Seafood', 'Deli', 'Cashier', 'Customer Service']
shift_statuses = ['Scheduled', 'Absent', 'On Leave']
store_names = ['ABC Queens East', 'ABC Queens West', 'ABC Brooklyn Central']
hours_options = [20, 30, 35, 40]  # to simulate part-time and full-time workers

# Prepare storage
employee_records = []
staffing_records = []

# Generate 100 employees
for _ in range(100):
    first_name = faker.first_name()
    last_name = faker.last_name()
    department = random.choice(departments)
    salary = round(random.uniform(2500, 6000), 2)
    hours_per_week = random.choice(hours_options)
    hire_date = faker.date_between(start_date='-5y', end_date='today')
    location_name = random.choice(store_names)

    # Add employee record
    employee_records.append({
        'first_name': first_name,
        'last_name': last_name,
        'department': department,
        'salary': salary,
        'hours_per_week': hours_per_week,
        'hire_date': hire_date,
        'location_name': location_name
    })

    # Decide number of shifts based on full-time/part-time
    if hours_per_week >= 35:
        num_shifts = random.randint(4, 5)  # full-time = more shifts
    else:
        num_shifts = random.randint(2, 3)  # part-time = fewer shifts

    for _ in range(num_shifts):
        shift_date = faker.date_between(start_date='-30d', end_date='+30d')
        start_time = random.choice(['08:00', '09:00', '10:00', '12:00'])
        end_time_options = {
            '08:00': '16:00',
            '09:00': '17:00',
            '10:00': '18:00',
            '12:00': '20:00'
        }
        end_time = end_time_options[start_time]
        shift_status = random.choices(shift_statuses, weights=[80, 10, 10])[0]  # 80% chance of Scheduled

        staffing_records.append({
            'first_name': first_name,
            'last_name': last_name,
            'shift_date': shift_date,
            'start_time': start_time,
            'end_time': end_time,
            'shift_status': shift_status,
            'location_name': location_name
        })

# Turn into DataFrames
df_employees = pd.DataFrame(employee_records)
df_staffing = pd.DataFrame(staffing_records)

# Save to Excel with two sheets
save_path = '/Users/kimminsung/Desktop/Columbia/SQL/SQL Final Project/ABC_employees.xlsx'
with pd.ExcelWriter(save_path) as writer:
    df_employees.to_excel(writer, sheet_name='Employees', index=False)
    df_staffing.to_excel(writer, sheet_name='Staffing', index=False)

print('Data generated successfully.')

Data generated successfully.


In [271]:
# Products and suppliers (500 records) 
from datetime import datetime

faker = Faker()

# Products and categories
products = [
    ("Large Eggs", "Dairy"),
    ("Organic Avocados", "Produce"),
    ("Classic Vanilla Ice Cream", "Frozen"),
    ("Broccoli", "Produce"),
    ("Ground Beef", "Meat"),
    ("Whole Milk", "Dairy"),
    ("Cheddar Cheese", "Dairy"),
    ("Frozen Pizza", "Frozen"),
    ("Chicken Breast", "Meat"),
    ("Apples", "Produce"),
]

suppliers = [faker.company() for _ in range(30)]
manufacturers = [faker.company() for _ in range(30)]

# Set number of rows
n_rows = 500

data = []
for _ in range(n_rows):
    productname, category = random.choice(products)
    manufacturer_name = random.choice(manufacturers)
    supplier_name = random.choice(suppliers)
    supplier_email = faker.unique.company_email()
    supplier_phone = faker.phone_number()
    notes = faker.sentence()

    # Pick a random date within the past 12 months
    order_date = faker.date_between(start_date='-1y', end_date='today')
    order_month = order_date.month

    # Generate base unit cost normally
    base_unit_cost = round(random.uniform(0.5, 20.0), 2)

    # Seasonal adjustments
    if "Egg" in product_name and order_month in [3, 4]:
        base_unit_cost *= 1.5
    if "Avocado" in product_name and order_month == 2:
        base_unit_cost *= 2
    if "Ice Cream" in product_name and order_month in [7, 8]:
        base_unit_cost *= 0.9

    base_unit_cost = round(base_unit_cost, 2)
    unit_price = round(base_unit_cost * random.uniform(1.2, 1.5), 2)
    expiration_days = random.randint(5, 90)

    data.append({
        "productname": productname,
        "category": category,
        "unitcost": base_unit_cost,
        "unitprice": unit_price,
        "expiration_days": expiration_days,
        "manufacturer_name": manufacturer_name,
        "supplier_name": supplier_name,
        "supplier_email": supplier_email,
        "supplier_phone": supplier_phone,
        "notes": notes
    })

# Save as Excel
df = pd.DataFrame(data)
pathfile = '/Users/kimminsung/Desktop/Columbia/SQL/SQL Final Project/ABC_products_and_suppliers.xlsx'
df.to_excel(pathfile, index=False)

print('Data generated successfully.')

Data generated successfully.


In [295]:
# Orders table (500 orders)
faker = Faker()

order_statuses = ['Ordered', 'Received', 'Cancelled']
store_names = ['ABC Queens East', 'ABC Queens West', 'ABC Brooklyn Central']
products = ['Large Eggs', 'Organic Avocados', 'Classic Vanilla Ice Cream', 'Ground Beef', 'Whole Milk',
            'Cheddar Cheese', 'Frozen Pizza', 'Chicken Breast','Apples']
suppliers_df = pd.read_excel('/Users/kimminsung/Desktop/Columbia/SQL/SQL Final Project/ABC_products_and_suppliers.xlsx')
suppliers = suppliers_df['supplier_name'].tolist()

records = []

for _ in range(500):
    order_date = faker.date_between(start_date='-6M', end_date='today')
    expected_delivery_date = order_date + pd.Timedelta(days=random.randint(1, 7))
    delivery_date = expected_delivery_date + pd.Timedelta(days=random.choice([0, 1, 2, -1]))
    supplier_name = random.choice(suppliers)
    location = random.choice(store_names)
    status = random.choice(order_statuses)
    productname = random.choice(products)
    units = random.randint(10, 200)
    unit_cost = round(random.uniform(1.0, 20.0), 2)

    records.append({
        'order_date': order_date,
        'expected_delivery_date': expected_delivery_date,
        'delivery_date': delivery_date if status != 'Ordered' else None,
        'supplier_name': supplier_name,
        'location': location,
        'status': status,
        'productname': productname,
        'units': units,
        'unit_cost': unit_cost
    })

df_orders = pd.DataFrame(records)
df_orders.to_excel('/Users/kimminsung/Desktop/Columbia/SQL/SQL Final Project/ABC_orders.xlsx', index=False)
print('Data generated successfully.')

Data generated successfully.


In [391]:
# Sales and customers table (1000 sales, included 3% of orders as 'returns')

faker = Faker()

products = ['Large Eggs', 'Organic Avocados', 'Classic Vanilla Ice Cream', 'Ground Beef', 'Whole Milk',
            'Cheddar Cheese', 'Frozen Pizza', 'Chicken Breast', 'Apples']
store_names = ['ABC Queens East', 'ABC Queens West', 'ABC Brooklyn Central']
genders = ['M', 'F', 'O']
promotion_names = ['Summer Sale', 'Winter Discount', 'Loyalty Reward', 'Flash Sale', None]

# Generate 250 customer profiles
num_customers = 250
customers = []

for _ in range(num_customers):
    customers.append({
        'customer_first_name': faker.first_name(),
        'customer_last_name': faker.last_name(),
        'age': random.randint(18, 80),
        'gender': random.choice(genders),
        'email': faker.email(),
        'location_name': random.choice(store_names),
        'loyalty_member': random.choice(['yes', 'no'])
    })

df_customers = pd.DataFrame(customers)
df_customers.to_excel('/Users/kimminsung/Desktop/Columbia/SQL/SQL Final Project/ABC_customers.xlsx', index=False)

# Step 2: Create 1000 sales, some linked to customers, some guest checkouts
sales_records = []

for _ in range(1000):
    sale_date = faker.date_between(start_date='-6M', end_date='today')
    location_name = random.choice(store_names)

    # Randomly assign customer (80% chance) or guest checkout (20%)
    if random.random() < 0.2:  # 20% guest checkouts
        customer = None
    else:
        customer = df_customers.sample(1).iloc[0]

    record = {
        'sale_date': sale_date,
        'location_name': location_name,
        'customer_first_name': None if customer is None else customer['customer_first_name'],
        'customer_last_name': None if customer is None else customer['customer_last_name'],
        'age': None if customer is None else customer['age'],
        'gender': None if customer is None else customer['gender'],
        'email': None if customer is None else customer['email'],
        'loyalty_member': None if customer is None else customer['loyalty_member'],
        'product_name': random.choice(products),
        'quantity': random.randint(1, 10),
        'unit_price': round(random.uniform(2.0, 50.0), 2),
        'discount_applied': random.choice([0, 5, 10, 15]),
        'promotion_name': random.choice(promotion_names)
    }
    sales_records.append(record)

df_sales_customers = pd.DataFrame(sales_records)

# Add "notes" column with 3% of the sales randomly marked as "Returned"
df_sales_customers['notes'] = None  # Initialize with None

# Randomly select 3% of the rows to be "Returned"
num_returns = int(len(df_sales_customers) * 0.03)
returned_indices = np.random.choice(df_sales_customers.index, size=num_returns, replace=False)

df_sales_customers.loc[returned_indices, 'notes'] = 'Returned'

df_sales_customers.to_excel('/Users/kimminsung/Desktop/Columbia/SQL/SQL Final Project/ABC_sales_and_customers.xlsx', index=False)

print('Data generated successfully!')

Data generated successfully!


In [433]:
# Expenses table (50 expenses)

faker = Faker()
store_names = ['ABC Queens East', 'ABC Queens West', 'ABC Brooklyn Central']
expense_categories = ['Utilities', 'Marketing', 'Rent', 'Insurance', 'Office Supplies', 'Miscellaneous']

# Generate 50 random expenses
records = []

for _ in range(50):
    category = random.choice(expense_categories)
    amount = round(random.uniform(100, 5000), 2)  # No bigger than 5000
    expense_date = faker.date_between(start_date='-6M', end_date='today')
    location_name = random.choice(store_names)
    description = f"{category} expense for {location_name}"

    records.append({
        'category': category,
        'amount': amount,
        'expense_date': expense_date,
        'location_name': location_name,
        'description': description
    })

# Step 2: Save as Excel
df_expenses = pd.DataFrame(records)
df_expenses.to_excel('/Users/kimminsung/Desktop/Columbia/SQL/SQL Final Project/ABC_accounting.xlsx', index=False)

print('Data generated successfully!')

Data generated successfully!


### Step 2: Data Cleaning and Manipulation

#### Employees Table
We will systematically inspect each file and make necessary transformations. The main steps for this file are:
1) Checking column names and renaming if necessary
2) Checking for missing required fields (such as first_name, last_name, location_name for NOT NULL constraint)
3) Format date types to datetime
4) Standardize categorical values (such as shift_status to be in 'scheduled','Absent','On Leave')
5) Check for relational integrity

In [35]:
import pandas as pd

# Read file and see the data first
file_path = '/Users/kimminsung/Desktop/Columbia/SQL/SQL Final Project/ABC_employees.xlsx'

# Read each sheet
employees_raw = pd.read_excel(file_path, sheet_name='Employees')
staffing_raw = pd.read_excel(file_path, sheet_name='Staffing')

# Display basic info
print("=== Employees Table ===")
print(employees_raw.info())
print(employees_raw.head())

print("\n=== Staffing Table ===")
print(staffing_raw.info())
print(staffing_raw.head())

=== Employees Table ===
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100 entries, 0 to 99
Data columns (total 7 columns):
 #   Column          Non-Null Count  Dtype         
---  ------          --------------  -----         
 0   first_name      100 non-null    object        
 1   last_name       100 non-null    object        
 2   department      100 non-null    object        
 3   salary          100 non-null    float64       
 4   hours_per_week  100 non-null    int64         
 5   hire_date       100 non-null    datetime64[ns]
 6   location_name   100 non-null    object        
dtypes: datetime64[ns](1), float64(1), int64(1), object(4)
memory usage: 5.6+ KB
None
  first_name last_name department   salary  hours_per_week  hire_date  \
0     Steven     Craig       Meat  2982.43              20 2022-09-14   
1    Patrick   Simmons     Bakery  3577.52              35 2023-08-09   
2    Jeffrey    Knight       Meat  4465.73              35 2022-03-19   
3    Felicia    Vaughn    P

In [39]:
# Step 1: Rename columns if needed (to match database schema)
employees_raw.columns = employees_raw.columns.str.strip().str.lower().str.replace(' ', '_')
staffing_raw.columns = staffing_raw.columns.str.strip().str.lower().str.replace(' ', '_')

# Step 2: Check for missing important values
print("Missing values in Employees:")
print(employees_raw[['first_name', 'last_name', 'location_name']].isnull().sum())

print("\nMissing values in Staffing:")
print(staffing_raw[['first_name', 'last_name', 'shift_date', 'start_time', 'end_time']].isnull().sum())

# Optional: Drop rows with missing critical values
employees_raw.dropna(subset=['first_name', 'last_name', 'location_name'], inplace=True)
staffing_raw.dropna(subset=['first_name', 'last_name', 'shift_date', 'start_time', 'end_time'], inplace=True)

# Step 3: Format datetime columns
employees_raw['hire_date'] = pd.to_datetime(employees_raw['hire_date'], errors='coerce')
staffing_raw['shift_date'] = pd.to_datetime(staffing_raw['shift_date'], errors='coerce')

# Step 4: Standardize categorical fields
staffing_raw['shift_status'] = staffing_raw['shift_status'].str.strip().str.capitalize()
allowed_statuses = ['Scheduled', 'Absent', 'On Leave']
staffing_raw = staffing_raw[staffing_raw['shift_status'].isin(allowed_statuses)]

# Final preview
print("\n=== Cleaned Employees Data ===")
print(employees_raw.head())

print("\n=== Cleaned Staffing Data ===")
print(staffing_raw.head())

Missing values in Employees:
first_name       0
last_name        0
location_name    0
dtype: int64

Missing values in Staffing:
first_name    0
last_name     0
shift_date    0
start_time    0
end_time      0
dtype: int64

=== Cleaned Employees Data ===
  first_name last_name department   salary  hours_per_week  hire_date  \
0     Steven     Craig       Meat  2982.43              20 2022-09-14   
1    Patrick   Simmons     Bakery  3577.52              35 2023-08-09   
2    Jeffrey    Knight       Meat  4465.73              35 2022-03-19   
3    Felicia    Vaughn    Produce  5286.28              40 2020-05-20   
4   Kimberly  Anderson       Deli  4114.70              40 2023-03-28   

          location_name  
0       ABC Queens West  
1  ABC Brooklyn Central  
2       ABC Queens West  
3       ABC Queens East  
4       ABC Queens West  

=== Cleaned Staffing Data ===
  first_name last_name shift_date start_time end_time shift_status  \
0     Steven     Craig 2025-05-08      09:00    17:

In [43]:
# Step 5: Checking relational integrity. We have to make sure that location_name matches store names in locations.
# Since there is no locations, we will create a reference table and map them.

locations_reference = pd.DataFrame({
    'location_id': [1, 2, 3],
    'location_name': ['ABC Queens East', 'ABC Queens West', 'ABC Brooklyn Central']
})

print(locations_reference)

# Merge locations into employees
employees_clean = employees_raw.merge(locations_reference, how='left', on='location_name')

# Merge locations into staffing
staffing_clean = staffing_raw.merge(locations_reference, how='left', on='location_name')

# Check if any locations failed to match
print("\nMissing locations in employees after merge:", employees_clean['location_id'].isnull().sum())
print("Missing locations in staffing after merge:", staffing_clean['location_id'].isnull().sum())

# Final cleaned datasets
print("\n=== Employees with location_id ===")
print(employees_clean[['first_name', 'last_name', 'location_name', 'location_id']].head())

print("\n=== Staffing with location_id ===")
print(staffing_clean[['first_name', 'last_name', 'location_name', 'location_id']].head())

   location_id         location_name
0            1       ABC Queens East
1            2       ABC Queens West
2            3  ABC Brooklyn Central

Missing locations in employees after merge: 0
Missing locations in staffing after merge: 0

=== Employees with location_id ===
  first_name last_name         location_name  location_id
0     Steven     Craig       ABC Queens West            2
1    Patrick   Simmons  ABC Brooklyn Central            3
2    Jeffrey    Knight       ABC Queens West            2
3    Felicia    Vaughn       ABC Queens East            1
4   Kimberly  Anderson       ABC Queens West            2

=== Staffing with location_id ===
  first_name last_name         location_name  location_id
0     Steven     Craig       ABC Queens West            2
1     Steven     Craig       ABC Queens West            2
2    Patrick   Simmons  ABC Brooklyn Central            3
3    Patrick   Simmons  ABC Brooklyn Central            3
4    Patrick   Simmons  ABC Brooklyn Central       

#### Products and suppliers table
In this table, we first have to map what information goes to which relation.

Mapping: ABC_products_and_suppliers.xlsx → Target Schema

| Spreadsheet Column  | Target Table   | Target Field        |
|:--------------------|:---------------|:--------------------|
| product_name         | products        | product_name         |
| category             | products        | category             |
| unit_cost            | products        | unit_cost            |
| unit_price           | products        | unit_price           |
| expiration_days      | products        | expiration_days      |
| manufacturer_name    | manufacturers   | manufacturer_name    |
| supplier_name        | suppliers       | supplier_name        |
| supplier_email       | suppliers       | email                |
| supplier_phone       | suppliers       | phone_number         |
| notes                | suppliers       | notes                |

Once mapped, the key tasks are as follows:
1) Ensuring no missing product names (No NULL values)
2) Removing any duplciates
3) Handling missing supplier or manufacturer emails or notes (Fill with NULL for missing values)
4) Normalize pricing (Ensure that the cost is higher than the price for sanity)
5) Assign IDs (for manufacturer_id, supplier_id) and link them into products

In [273]:
import pandas as pd

# Load the raw products and suppliers spreadsheet
products_suppliers_raw = pd.read_excel('/Users/kimminsung/Desktop/Columbia/SQL/SQL Final Project/ABC_products_and_suppliers.xlsx')

# View basic info
print(products_suppliers_raw.info())  # Check datatypes and missing values
print(products_suppliers_raw.head())  # View sample rows

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 500 entries, 0 to 499
Data columns (total 10 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   productname        500 non-null    object 
 1   category           500 non-null    object 
 2   unitcost           500 non-null    float64
 3   unitprice          500 non-null    float64
 4   expiration_days    500 non-null    int64  
 5   manufacturer_name  500 non-null    object 
 6   supplier_name      500 non-null    object 
 7   supplier_email     500 non-null    object 
 8   supplier_phone     500 non-null    object 
 9   notes              500 non-null    object 
dtypes: float64(2), int64(1), object(7)
memory usage: 39.2+ KB
None
                 productname category  unitcost  unitprice  expiration_days  \
0                Ground Beef     Meat      2.78       3.51               27   
1               Frozen Pizza   Frozen      6.95       8.51               51   
2           Or

In [275]:
# Cleaning/standardizing

# Rename columns so they match the schema if needed
products_suppliers_clean = products_suppliers_raw.rename(columns={
    'productname': 'product_name',
    'category': 'category',
    'unitcost': 'unit_cost',
    'unitprice': 'unit_price',
    'expiration_days': 'expiration_days',
    'manufacturer_name': 'manufacturer_name',
    'supplier_name': 'supplier_name',
    'supplier_email': 'supplier_email',
    'supplier_phone': 'supplier_phone',   
    'notes': 'notes'
})

# Step 4: Handle missing values (example: if unit_cost or unit_price is missing, flag it)
products_suppliers_clean['unit_cost'] = products_suppliers_clean['unit_cost'].fillna(0)
products_suppliers_clean['unit_price'] = products_suppliers_clean['unit_price'].fillna(0)
products_suppliers_clean['expiration_days'] = products_suppliers_clean['expiration_days'].fillna(30)  # Assume 30 days if missing

# Step 5: (Optional) Drop fully empty supplier fields if needed
products_suppliers_clean = products_suppliers_clean.dropna(subset=['supplier_name'])

# Step 6: Review cleaned data
print(products_suppliers_clean.info())
print(products_suppliers_clean.head())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 500 entries, 0 to 499
Data columns (total 10 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   product_name       500 non-null    object 
 1   category           500 non-null    object 
 2   unit_cost          500 non-null    float64
 3   unit_price         500 non-null    float64
 4   expiration_days    500 non-null    int64  
 5   manufacturer_name  500 non-null    object 
 6   supplier_name      500 non-null    object 
 7   supplier_email     500 non-null    object 
 8   supplier_phone     500 non-null    object 
 9   notes              500 non-null    object 
dtypes: float64(2), int64(1), object(7)
memory usage: 39.2+ KB
None
                product_name category  unit_cost  unit_price  expiration_days  \
0                Ground Beef     Meat       2.78        3.51               27   
1               Frozen Pizza   Frozen       6.95        8.51               51   
2       

#### Orders table
Again, we will first map the table to the target schema.

Mapping for ABC_orders.xlsx -> Target Schema

| Spreadsheet Column         | Target Table(s)                  | Target Field                      |
|-----------------------------|-----------------------------------|------------------------------------|
| order_date                  | purchase_orders                   | order_date                        |
| expected_delivery_date      | deliveries                        | expected_delivery_date            |
| delivery_date               | deliveries                        | delivery_date                     |
| supplier_name               | purchase_orders & deliveries      | supplier_id (after lookup)        |
| location_name               | purchase_orders                   | location_id (after lookup)        |
| status                      | purchase_orders                   | status                            |
| product_name                | purchase_order_details            | product_id (after lookup)         |
| quantity                    | purchase_order_details            | quantity                          |
| unit_cost                   | purchase_order_details            | unit_cost                         |

Once mapped, the key tasks are as follows:
1) Mapping client status to schema (pending, received -> ordered, received, cancelled)
2) Fix inconsistent supplier and location names
3) Convert dates into proper datetime format
4) Product mapping between product name and product_id from products table

In [297]:
# Loading and displaying info
import pandas as pd

orders_raw = pd.read_excel('/Users/kimminsung/Desktop/Columbia/SQL/SQL Final Project/ABC_orders.xlsx')

print(orders_raw.info())
print(orders_raw.head())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 500 entries, 0 to 499
Data columns (total 9 columns):
 #   Column                  Non-Null Count  Dtype         
---  ------                  --------------  -----         
 0   order_date              500 non-null    datetime64[ns]
 1   expected_delivery_date  500 non-null    datetime64[ns]
 2   delivery_date           341 non-null    datetime64[ns]
 3   supplier_name           500 non-null    object        
 4   location                500 non-null    object        
 5   status                  500 non-null    object        
 6   productname             500 non-null    object        
 7   units                   500 non-null    int64         
 8   unit_cost               500 non-null    float64       
dtypes: datetime64[ns](3), float64(1), int64(1), object(4)
memory usage: 35.3+ KB
None
  order_date expected_delivery_date delivery_date  supplier_name  \
0 2025-04-07             2025-04-08    2025-04-09     Warner LLC   
1 2024-12-04 

In [299]:
# Clean the column names
orders_clean = orders_raw.rename(columns={
    'order_date': 'order_date',
    'expected_delivery_date': 'expected_delivery_date',
    'delivery_date': 'delivery_date',
    'supplier_name': 'supplier_name',
    'location': 'location_name',
    'status': 'status',
    'productname': 'product_name',
    'units': 'quantity',
    'unit_cost': 'unit_cost'
})

# Step 4: Map statuses to correct schema values
status_mapping = {
    'pending': 'Ordered',
    'shipped': 'Ordered',
    'completed': 'Received',
    'received': 'Received',
    'cancelled': 'Cancelled',
    'canceled': 'Cancelled',
    'delivered': 'Received'
}
orders_clean['status'] = orders_clean['status'].str.lower().map(status_mapping).fillna('Ordered')

# Step 5: Convert date fields
orders_clean['order_date'] = pd.to_datetime(orders_clean['order_date'], errors='coerce')
orders_clean['expected_delivery_date'] = pd.to_datetime(orders_clean['expected_delivery_date'], errors='coerce')
orders_clean['delivery_date'] = pd.to_datetime(orders_clean['delivery_date'], errors='coerce')

# Step 6: Fill missing quantities and costs
orders_clean['quantity'] = orders_clean['quantity'].fillna(1)
orders_clean['unit_cost'] = orders_clean['unit_cost'].fillna(0)

# Step 7: Create clean DataFrames

# 7.1 Purchase Orders (deduplicated at order level)
purchase_orders_df = orders_clean[['order_date', 'supplier_name', 'location_name', 'status']].drop_duplicates()

# 7.2 Deliveries (deduplicated at order level)
deliveries_df = orders_clean[['order_date', 'expected_delivery_date', 'delivery_date', 'supplier_name']].drop_duplicates()

# 7.3 Purchase Order Details (full product-level granularity)
purchase_order_details_df = orders_clean[['product_name', 'quantity', 'unit_cost']]

# Step 8: Check cleaned tables
print(purchase_orders_df.head())
print(deliveries_df.head())
print(purchase_order_details_df.head())

  order_date  supplier_name         location_name     status
0 2025-04-07     Warner LLC       ABC Queens West   Received
1 2024-12-04    Jackson Ltd       ABC Queens West  Cancelled
2 2025-03-28    Smith Group  ABC Brooklyn Central   Received
3 2025-04-12    Jackson Ltd       ABC Queens East   Received
4 2024-12-20  Watson-Weaver       ABC Queens East    Ordered
  order_date expected_delivery_date delivery_date  supplier_name
0 2025-04-07             2025-04-08    2025-04-09     Warner LLC
1 2024-12-04             2024-12-06    2024-12-06    Jackson Ltd
2 2025-03-28             2025-04-01    2025-03-31    Smith Group
3 2025-04-12             2025-04-18    2025-04-20    Jackson Ltd
4 2024-12-20             2024-12-24           NaT  Watson-Weaver
     product_name  quantity  unit_cost
0     Ground Beef        69      18.70
1      Whole Milk       107      11.99
2  Cheddar Cheese       168       3.36
3      Large Eggs       168       4.46
4      Large Eggs        12       7.54


#### Sales and customers table

Mapping for ABC_sales_and_customers.xlsx

| Spreadsheet Column     | Target Table(s)         | Target Field                        |
|-------------------------|-------------------------|-------------------------------------|
| sale_date               | sales_orders             | order_date                          |
| location_name           | sales_orders             | location_id (after lookup)          |
| customer_first_name     | customers                | first_name                          |
| customer_last_name      | customers                | last_name                           |
| age                     | customers                | age                                 |
| gender                  | customers                | gender                              |
| email                   | customers                | email                               |
| loyalty_member          | customers                | loyalty_member (map 'yes'/'no' to boolean) |
| product_name            | sale_order_details       | product_id (after lookup)           |
| quantity                | sale_order_details       | quantity                            |
| unit_price              | sale_order_details       | unit_price                          |
| discount_applied        | sale_order_details       | discount                            |
| promotion_name          | promotions (after mapping) | promotion_id (if available)        |

Key tasks:
1) Change loyalty member from yes/no to true/false
2) Handle missing values for promotions
3) Create sales order id (unique combination of sale_date, location, customer)

In [393]:
import pandas as pd

# Load the raw sales data
sales_customers = pd.read_excel('/Users/kimminsung/Desktop/Columbia/SQL/SQL Final Project/ABC_sales_and_customers.xlsx')

# Preview the dataset
print(sales_customers.info())
print(sales_customers.head())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 13 columns):
 #   Column               Non-Null Count  Dtype         
---  ------               --------------  -----         
 0   sale_date            1000 non-null   datetime64[ns]
 1   location_name        1000 non-null   object        
 2   customer_first_name  825 non-null    object        
 3   customer_last_name   825 non-null    object        
 4   age                  825 non-null    float64       
 5   gender               825 non-null    object        
 6   email                825 non-null    object        
 7   loyalty_member       825 non-null    object        
 8   product_name         1000 non-null   object        
 9   quantity             1000 non-null   int64         
 10  unit_price           1000 non-null   float64       
 11  discount_applied     1000 non-null   int64         
 12  promotion_name       794 non-null    object        
dtypes: datetime64[ns](1), float64(2), 

In [395]:
# Step 1: Standardize loyalty_member column
sales_customers['loyalty_member'] = sales_customers['loyalty_member'].map({
    'yes': True,
    'no': False
})

# Step 2: Fill missing discount_applied with 0
sales_customers['discount_applied'] = sales_customers['discount_applied'].fillna(0)

# Step 3: Create unique sale_order_id
sales_customers['sale_order_id'] = range(1, len(sales_customers) + 1)

print("\nCleaned dataset:")
print(sales_customers.head())


Cleaned dataset:
   sale_date         location_name customer_first_name customer_last_name  \
0 2024-12-04       ABC Queens East               Erica              Evans   
1 2024-11-06  ABC Brooklyn Central                 NaN                NaN   
2 2024-11-29       ABC Queens West                Gail              Perry   
3 2025-02-01  ABC Brooklyn Central                 NaN                NaN   
4 2025-02-01  ABC Brooklyn Central            Jonathan         Williamson   

    age gender                     email loyalty_member  \
0  71.0      O      joseph37@example.net          False   
1   NaN    NaN                       NaN            NaN   
2  51.0      F  thomassandra@example.com          False   
3   NaN    NaN                       NaN            NaN   
4  63.0      F   davidthomas@example.net           True   

                product_name  quantity  unit_price  discount_applied  \
0                     Apples         9       10.60                 0   
1                   

#### Accounting table
After checking with the finance team of the client, the accounting table matches the schema and will require no pre-processing. Due to the nature of needing to document all expenses, we confirmed with the team that there are no NULL values. The only manipulation will come in the loading phase to change the location_name with the location_id.

### Step 3: Data Loading

We will now load the migrate the cleaned datasets into our schema.
Here are the key tasks required:
1) Save each excel file separately as csv files
2) Insert current data considering dependencies in order
    * Insert data into foundation tables (no dependencies)
    * Insert data into product & supply chains
    * Insert data into sales and promotions
    * Insert data into financials and returns tables
    * Check if triggers run correctly

In [435]:
# Changing Excel to csv files
import pandas as pd

# Load Excel files
employees_xls = pd.read_excel('/Users/kimminsung/Desktop/Columbia/SQL/SQL Final Project/ABC_employees.xlsx', sheet_name=None)
products_suppliers_xls = pd.read_excel('/Users/kimminsung/Desktop/Columbia/SQL/SQL Final Project/ABC_products_and_suppliers.xlsx')
orders_xls = pd.read_excel('/Users/kimminsung/Desktop/Columbia/SQL/SQL Final Project/ABC_orders.xlsx')
sales_customers_xls = pd.read_excel('/Users/kimminsung/Desktop/Columbia/SQL/SQL Final Project/ABC_sales_and_customers.xlsx')
accounting_xls = pd.read_excel('/Users/kimminsung/Desktop/Columbia/SQL/SQL Final Project/ABC_accounting.xlsx')


# Save as CSVs
for sheet_name, df in employees_xls.items():
    df.to_csv(f'/Users/kimminsung/Desktop/Columbia/SQL/SQL Final Project/ABC_employees_{sheet_name}.csv', index=False)

products_suppliers_xls.to_csv('/Users/kimminsung/Desktop/Columbia/SQL/SQL Final Project/ABC_products_and_suppliers.csv', index=False)
orders_xls.to_csv('/Users/kimminsung/Desktop/Columbia/SQL/SQL Final Project/ABC_orders.csv', index=False)
sales_customers_xls.to_csv('/Users/kimminsung/Desktop/Columbia/SQL/SQL Final Project/ABC_sales_and_customers.csv', index=False)
accounting_xls.to_csv('/Users/kimminsung/Desktop/Columbia/SQL/SQL Final Project/ABC_accounting.csv')

print("All Excel files converted to CSV successfully.")

All Excel files converted to CSV successfully.


We then loaded all the data into each table.

In [283]:
# 1) Locations

conn = psycopg2.connect(
    dbname='sql_final_project',
    user='postgres',
    password='123',
    host='localhost',
    port='5432'
)
cur = conn.cursor()

employees_raw = pd.read_csv('/Users/kimminsung/Desktop/Columbia/SQL/SQL Final Project/ABC_employees_Employees.csv')

# Extract unique location names
unique_locations = employees_raw['location_name'].dropna().unique()

# Insert unique locations
for loc in unique_locations:
    cur.execute("""
        INSERT INTO locations (location_name, address, city, state, zip_code, opened_date, type)
        VALUES (%s, NULL, NULL, NULL, NULL, NULL, 'Other')
        ON CONFLICT (location_name) DO NOTHING;
    """, (loc,))
conn.commit()

print("Unique locations inserted successfully.")

# 5. Close cursor (optional now)
cur.close()
conn.close()

Unique locations inserted successfully.


In [285]:
# 2) Loading suppliers

products_suppliers_cleaned = pd.read_csv('/Users/kimminsung/Desktop/Columbia/SQL/SQL Final Project/ABC_products_and_suppliers.csv')

# Connect to PostgreSQL
conn = psycopg2.connect(
    dbname="sql_final_project",
    user="postgres",
    password="123",
    host="localhost",
    port="5432"
)
cur = conn.cursor()

# Get unique suppliers
unique_suppliers = products_suppliers_cleaned[['supplier_name', 'supplier_email', 'supplier_phone', 'notes']].drop_duplicates()

# Insert each supplier into suppliers table
for index, row in unique_suppliers.iterrows():
    supplier_name = row['supplier_name'][:100] if pd.notna(row['supplier_name']) else None
    email = row['supplier_email'][:100] if pd.notna(row['supplier_email']) else None
    phone_number = row['supplier_phone'][:20] if pd.notna(row['supplier_phone']) else None
    notes = row['notes'] if pd.notna(row['notes']) else None  # Text field, no need to truncate
    
    cur.execute("""
        INSERT INTO suppliers (supplier_name, email, phone_number, notes)
        VALUES (%s, %s, %s, %s);
    """, (
        supplier_name,
        email,
        phone_number,
        notes
    ))

conn.commit()
cur.close()
conn.close()

print("Suppliers inserted successfully!")

Suppliers inserted successfully!


In [287]:
# 3) Loading manufacturers

conn = psycopg2.connect(
    dbname="sql_final_project",
    user="postgres",
    password="123",
    host="localhost",
    port="5432"
)
cur = conn.cursor()

# Load manufacturers data from the products_and_suppliers CSV
products_suppliers_df = pd.read_csv('/Users/kimminsung/Desktop/Columbia/SQL/SQL Final Project/ABC_products_and_suppliers.csv')

# Extract and deduplicate manufacturer names
unique_manufacturers = products_suppliers_df[['manufacturer_name']].drop_duplicates()

# Insert each manufacturer into the 'manufacturers' table
for idx, row in unique_manufacturers.iterrows():
    cur.execute("""
        INSERT INTO manufacturers (manufacturer_name)
        VALUES (%s)
    """, (row['manufacturer_name'],))

# Commit and close
conn.commit()
cur.close()
conn.close()

print("Manufacturers inserted successfully.")

Manufacturers inserted successfully.


In [291]:
# 4) Loading Products

conn = psycopg2.connect(
    dbname="sql_final_project",
    user="postgres",
    password="123",
    host="localhost",
    port="5432"
)
cur = conn.cursor()

# Load cleaned products and suppliers data
products_suppliers = pd.read_csv('/Users/kimminsung/Desktop/Columbia/SQL/SQL Final Project/ABC_products_and_suppliers.csv')

# Insert products (without inserting product_id)
for index, row in products_suppliers.iterrows():
    # Lookup manufacturer_id
    cur.execute("SELECT manufacturer_id FROM manufacturers WHERE manufacturer_name = %s;", (row['manufacturer_name'],))
    manufacturer_result = cur.fetchone()
    
    # Lookup supplier_id
    cur.execute("SELECT supplier_id FROM suppliers WHERE supplier_name = %s;", (row['supplier_name'],))
    supplier_result = cur.fetchone()

    # Safety checks
    if manufacturer_result is None:
        print(f"Manufacturer not found for {row['manufacturer_name']} (skipping row)")
        continue
    if supplier_result is None:
        print(f"Supplier not found for {row['supplier_name']} (skipping row)")
        continue
    
    manufacturer_id = manufacturer_result[0]
    supplier_id = supplier_result[0]

    # Insert into products (no product_id)
    cur.execute("""
        INSERT INTO products (product_name, category, unit_cost, unit_price, expiration_days, manufacturer_id, supplier_id)
        VALUES (%s, %s, %s, %s, %s, %s, %s);
    """, (
        row['product_name'],
        row['category'],
        row['unit_cost'],
        row['unit_price'],
        row['expiration_days'],
        manufacturer_id,
        supplier_id
    ))

conn.commit()
cur.close()
conn.close()

print("Products inserted successfully!")

Products inserted successfully!


In [305]:
# 5) Loading purchase orders

orders_clean = pd.read_csv('/Users/kimminsung/Desktop/Columbia/SQL/SQL Final Project/ABC_orders.csv')

# Connect to PostgreSQL
conn = psycopg2.connect(
    dbname='sql_final_project',
    user='postgres',
    password='123',
    host='localhost',
    port='5432'
)
cur = conn.cursor()

for index, row in orders_clean.iterrows():
    # Lookup supplier_id
    cur.execute("SELECT supplier_id FROM suppliers WHERE supplier_name = %s", (row['supplier_name'],))
    supplier_result = cur.fetchone()
    if supplier_result is None:
        print(f"Warning: Supplier '{row['supplier_name']}' not found, skipping this row.")
        continue
    supplier_id = supplier_result[0]

    # Lookup location_id
    cur.execute("SELECT location_id FROM locations WHERE location_name = %s", (row['location_name'],))
    location_result = cur.fetchone()
    if location_result is None:
        print(f"Warning: Location '{row['location_name']}' not found, skipping this row.")
        continue
    location_id = location_result[0]

    # Prepare delivery_date safely
    delivery_date = None if pd.isna(row['delivery_date']) else row['delivery_date']

    # Insert into purchase_orders
    try:
        cur.execute("""
            INSERT INTO purchase_orders (supplier_id, location_id, order_date, delivery_date, status)
            VALUES (%s, %s, %s, %s, %s)
        """, (
            supplier_id,
            location_id,
            row['order_date'],
            delivery_date,   # This will be None if NaN
            row['status']
        ))
    except Exception as e:
        print(f"Error inserting row {index}: {e}")

conn.commit()
cur.close()
conn.close()

print("Purchase orders inserted successfully!")

Purchase orders inserted successfully!


In [307]:
# 6) Deliveries

orders_clean = pd.read_csv('/Users/kimminsung/Desktop/Columbia/SQL/SQL Final Project/ABC_orders.csv')

conn = psycopg2.connect(
    dbname='sql_final_project',
    user='postgres',
    password='123',
    host='localhost',
    port='5432'
)
cur = conn.cursor()

# Insert each delivery
for index, row in orders_clean.iterrows():
    # Lookup supplier_id
    cur.execute("SELECT supplier_id FROM suppliers WHERE supplier_name = %s", (row['supplier_name'],))
    supplier_result = cur.fetchone()
    if supplier_result is None:
        print(f"Warning: Supplier '{row['supplier_name']}' not found, skipping this row.")
        continue
    supplier_id = supplier_result[0]

    # Lookup purchase_order_id
    cur.execute("""
        SELECT purchase_order_id FROM purchase_orders 
        WHERE supplier_id = %s AND order_date = %s
    """, (supplier_id, row['order_date']))
    purchase_order_result = cur.fetchone()
    if purchase_order_result is None:
        print(f"Warning: Purchase order not found for supplier '{row['supplier_name']}' on {row['order_date']}, skipping.")
        continue
    purchase_order_id = purchase_order_result[0]

    # Insert into deliveries
    try:
        cur.execute("""
            INSERT INTO deliveries (purchase_order_id, supplier_id, expected_delivery_date, delivery_date, status)
            VALUES (%s, %s, %s, %s, %s)
        """, (
            purchase_order_id,
            supplier_id,
            row['expected_delivery_date'],
            None if pd.isna(row['delivery_date']) else row['delivery_date'],
            'Scheduled' if row['status'] == 'Ordered' else 
            'Delivered' if row['status'] == 'Received' else
            'Delayed'
        ))
    except Exception as e:
        print(f"Error inserting delivery row {index}: {e}")

conn.commit()
cur.close()
conn.close()

print("Deliveries inserted successfully!")

Deliveries inserted successfully!


In [309]:
# 7) Loading purchase order details

orders_clean = pd.read_csv('/Users/kimminsung/Desktop/Columbia/SQL/SQL Final Project/ABC_orders.csv')

conn = psycopg2.connect(
    dbname='sql_final_project',
    user='postgres',
    password='123',
    host='localhost',
    port='5432'
)
cur = conn.cursor()

for index, row in orders_clean.iterrows():
    # Find purchase_order_id using JOINs
    cur.execute("""
        SELECT po.purchase_order_id
        FROM purchase_orders po
        JOIN suppliers s ON po.supplier_id = s.supplier_id
        JOIN locations l ON po.location_id = l.location_id
        WHERE s.supplier_name = %s
          AND l.location_name = %s
          AND po.order_date = %s
        LIMIT 1;
    """, (row['supplier_name'], row['location_name'], row['order_date']))
    purchase_order_result = cur.fetchone()
    
    if purchase_order_result is None:
        print(f"Purchase order not found for row {index}. Skipping.")
        continue
    purchase_order_id = purchase_order_result[0]

    # Lookup product_id
    cur.execute("""
        SELECT product_id
        FROM products
        WHERE product_name = %s
        LIMIT 1;
    """, (row['product_name'],))
    product_result = cur.fetchone()
    if product_result is None:
        print(f"Product '{row['product_name']}' not found for row {index}. Skipping.")
        continue
    product_id = product_result[0]

    # Insert into purchase_order_details
    try:
        cur.execute("""
            INSERT INTO purchase_order_details (purchase_order_id, product_id, quantity, unit_cost)
            VALUES (%s, %s, %s, %s);
        """, (
            purchase_order_id,
            product_id,
            row['units'],
            row['unit_cost']
        ))
    except Exception as e:
        print(f"Error inserting row {index}: {e}")

conn.commit()
cur.close()
conn.close()

print("Purchase order details inserted successfully!")

Purchase order details inserted successfully!


In [323]:
# 8) Loading inventory

conn = psycopg2.connect(
    dbname='sql_final_project',
    user='postgres',
    password='123',
    host='localhost',
    port='5432'
)
cur = conn.cursor()

# Load the orders CSV
purchase_orders = pd.read_csv('/Users/kimminsung/Desktop/Columbia/SQL/SQL Final Project/ABC_orders.csv')

for index, row in purchase_orders.iterrows():
    if row['status'] != 'Received':
        continue

    # Find supplier_id
    cur.execute("SELECT supplier_id FROM suppliers WHERE supplier_name = %s LIMIT 1", (row['supplier_name'],))
    supplier_result = cur.fetchone()
    if supplier_result is None:
        print(f"Warning: Supplier '{row['supplier_name']}' not found.")
        continue
    supplier_id = supplier_result[0]

    # Find location_id
    cur.execute("SELECT location_id FROM locations WHERE location_name = %s LIMIT 1", (row['location_name'],))
    location_result = cur.fetchone()
    if location_result is None:
        print(f"Warning: Location '{row['location_name']}' not found.")
        continue
    location_id = location_result[0]

    # Find product_id
    cur.execute("SELECT product_id FROM products WHERE product_name = %s LIMIT 1", (row['product_name'],))
    product_result = cur.fetchone()
    if product_result is None:
        print(f"Warning: Product '{row['product_name']}' not found.")
        continue
    product_id = product_result[0]

    # Step 4: Find purchase_order_id
    cur.execute("""
        SELECT purchase_order_id
        FROM purchase_orders
        WHERE supplier_id = %s
        AND location_id = %s
        AND order_date = %s
        LIMIT 1
    """, (supplier_id, location_id, row['order_date']))
    purchase_order_result = cur.fetchone()
    if purchase_order_result is None:
        print(f"Warning: Purchase Order not found for supplier '{row['supplier_name']}' at '{row['location_name']}' on '{row['order_date']}'.")
        continue
    purchase_order_id = purchase_order_result[0]

    # Find delivery_id
    cur.execute("""
        SELECT delivery_id
        FROM deliveries
        WHERE purchase_order_id = %s
        LIMIT 1
    """, (purchase_order_id,))
    delivery_result = cur.fetchone()
    if delivery_result is None:
        print(f"Warning: Delivery not found for PO ID {purchase_order_id}.")
        continue
    delivery_id = delivery_result[0]

    # Calculate expiration date
    cur.execute("SELECT expiration_days FROM products WHERE product_id = %s", (product_id,))
    expiration_days_result = cur.fetchone()
    expiration_days = expiration_days_result[0] if expiration_days_result else None

    if expiration_days:
        cur.execute("SELECT delivery_date FROM deliveries WHERE delivery_id = %s", (delivery_id,))
        delivery_date_result = cur.fetchone()
        if delivery_date_result and delivery_date_result[0]:
            delivery_date = delivery_date_result[0]
            expiration_date = delivery_date + pd.Timedelta(days=expiration_days)
        else:
            expiration_date = None
    else:
        expiration_date = None

    # Insert into inventory
    try:
        cur.execute("""
            INSERT INTO inventory (product_id, location_id, quantity, entry_date, expiration_date, purchase_order_id, delivery_id)
            VALUES (%s, %s, %s, %s, %s, %s, %s)
        """, (
            product_id,
            location_id,
            row['units'],
            row['delivery_date'],
            expiration_date,
            purchase_order_id,
            delivery_id
        ))
    except Exception as e:
        print(f"Error inserting inventory row {index}: {e}")
        conn.rollback()
        continue

conn.commit()
cur.close()
conn.close()

print("Inventory inserted successfully!")

Inventory inserted successfully!


Some values were skipped for purchase orders as the delivery date might still not be delivered.

In [328]:
# 9) Employees

employees_clean = pd.read_csv('/Users/kimminsung/Desktop/Columbia/SQL/SQL Final Project/ABC_employees_Employees.csv')

conn = psycopg2.connect(
    dbname='sql_final_project',
    user='postgres',
    password='123',
    host='localhost',
    port='5432'
)
cur = conn.cursor()

for index, row in employees_clean.iterrows():
    # Lookup location_id based on location_name
    cur.execute("SELECT location_id FROM locations WHERE location_name = %s", (row['location_name'],))
    location_result = cur.fetchone()
    if location_result is None:
        print(f"Warning: Location '{row['location_name']}' not found, skipping employee {row['first_name']} {row['last_name']}.")
        continue
    location_id = location_result[0]

    # Insert into employees
    cur.execute("""
        INSERT INTO employees (location_id, first_name, last_name, department, salary, hours_per_week, hire_date, notes)
        VALUES (%s, %s, %s, %s, %s, %s, %s, %s)
    """, (
        location_id,
        row['first_name'],
        row['last_name'],
        row['department'],
        row['salary'],
        row['hours_per_week'],
        row['hire_date'],
        None  # notes column, we don't have it in data now
    ))

conn.commit()
cur.close()
conn.close()

print("Employees inserted successfully!")

Employees inserted successfully!


In [340]:
# 10) Staffing

staffing_clean = pd.read_excel('/Users/kimminsung/Desktop/Columbia/SQL/SQL Final Project/ABC_employees.xlsx', sheet_name='Staffing')

conn = psycopg2.connect(
    dbname='sql_final_project',
    user='postgres',
    password='123',
    host='localhost',
    port='5432'
)
cur = conn.cursor()

for index, row in staffing_clean.iterrows():
    # Find employee_id using first_name + last_name
    cur.execute("""
        SELECT employee_id 
        FROM employees
        WHERE first_name = %s AND last_name = %s
        LIMIT 1
    """, (row['first_name'], row['last_name']))
    employee_result = cur.fetchone()

    if employee_result is None:
        print(f"Warning: Employee '{row['first_name']} {row['last_name']}' not found, skipping.")
        continue
    employee_id = employee_result[0]

    # Lookup location_id
    cur.execute("SELECT location_id FROM locations WHERE location_name = %s", (row['location_name'],))
    location_result = cur.fetchone()
    if location_result is None:
        print(f"Warning: Location '{row['location_name']}' not found for shift, skipping.")
        continue
    location_id = location_result[0]

    # Insert into staffing table
    cur.execute("""
        INSERT INTO staffing (employee_id, location_id, shift_date, start_time, end_time, status)
        VALUES (%s, %s, %s, %s, %s, %s)
    """, (
        employee_id,
        location_id,
        row['shift_date'],
        row['start_time'],
        row['end_time'],
        row['shift_status']
    ))

conn.commit()
cur.close()
conn.close()

print("Staffing inserted successfully!")

Staffing inserted successfully!


In [401]:
# 11) Customers

conn = psycopg2.connect(
    dbname='sql_final_project',
    user='postgres',
    password='123',
    host='localhost',
    port='5432'
)
cur = conn.cursor()

sales_customers_clean = pd.read_csv('/Users/kimminsung/Desktop/Columbia/SQL/SQL Final Project/ABC_sales_and_customers.csv')

for index, row in sales_customers_clean.iterrows():
    # Lookup location_id from location_name
    cur.execute("SELECT location_id FROM locations WHERE location_name = %s", (row['location_name'],))
    location_result = cur.fetchone()
    if location_result is None:
        print(f"Warning: Location '{row['location_name']}' not found, skipping.")
        continue
    location_id = location_result[0]

    # Clean missing customer info
    first_name = row['customer_first_name'] if pd.notna(row['customer_first_name']) else None
    last_name = row['customer_last_name'] if pd.notna(row['customer_last_name']) else None
    age = int(row['age']) if pd.notna(row['age']) else None
    gender = row['gender'] if pd.notna(row['gender']) else None
    email = row['email'] if pd.notna(row['email']) else None

    # Skip inserting if no first name or last name
    if first_name is None or last_name is None:
        print(f"Skipping row {index}: missing customer name.")
        continue

    # Loyalty handling
    if pd.isna(row['loyalty_member']):
        loyalty_member = False
    else:
        loyalty_member = True if str(row['loyalty_member']).lower() == 'yes' else False

    # Insert into customers
    cur.execute("""
        INSERT INTO customers (first_name, last_name, age, gender, email, location_id, loyalty_member)
        VALUES (%s, %s, %s, %s, %s, %s, %s)
    """, (
        first_name,
        last_name,
        age,
        gender,
        email,
        location_id,
        loyalty_member
    ))

conn.commit()
cur.close()
conn.close()

print("Customers inserted successfully!")

Skipping row 1: missing customer name.
Skipping row 3: missing customer name.
Skipping row 9: missing customer name.
Skipping row 36: missing customer name.
Skipping row 49: missing customer name.
Skipping row 50: missing customer name.
Skipping row 63: missing customer name.
Skipping row 76: missing customer name.
Skipping row 80: missing customer name.
Skipping row 89: missing customer name.
Skipping row 90: missing customer name.
Skipping row 93: missing customer name.
Skipping row 102: missing customer name.
Skipping row 104: missing customer name.
Skipping row 105: missing customer name.
Skipping row 110: missing customer name.
Skipping row 114: missing customer name.
Skipping row 123: missing customer name.
Skipping row 126: missing customer name.
Skipping row 127: missing customer name.
Skipping row 129: missing customer name.
Skipping row 136: missing customer name.
Skipping row 146: missing customer name.
Skipping row 154: missing customer name.
Skipping row 163: missing custo

Not all buyers are customers (that gave their information and joined the loyalty program). Some those that do not have a name will be dropped.

In [408]:
# 12) Promotions

sales_customers = pd.read_csv('/Users/kimminsung/Desktop/Columbia/SQL/SQL Final Project/ABC_sales_and_customers.csv')

# Only keep rows with a promotion name
sales_promotions = sales_customers[sales_customers['promotion_name'].notnull()]

# Aggregate
promotion_summary = sales_promotions.groupby('promotion_name').agg({
    'discount_applied': 'mean',
    'sale_date': ['min', 'max']
}).reset_index()

# Flatten multi-index columns
promotion_summary.columns = ['promotion_name', 'avg_discount', 'start_date', 'end_date']

conn = psycopg2.connect(
    dbname='sql_final_project',
    user='postgres',
    password='123',
    host='localhost',
    port='5432'
)
cur = conn.cursor()

cur.execute("SELECT location_id FROM locations;")
location_ids = [row[0] for row in cur.fetchall()]

# Step 6: Insert promotions
for idx, row in promotion_summary.iterrows():
    promotion_name = row['promotion_name']
    avg_discount = round(row['avg_discount'], 2)
    start_date = row['start_date']
    end_date = row['end_date']
    location_id = random.choice(location_ids) 
    
    cur.execute("""
        INSERT INTO promotions (promotion_name, start_date, end_date, location_id, discount_percentage, loyalty_only)
        VALUES (%s, %s, %s, %s, %s, %s);
    """, (
        promotion_name,
        start_date,
        end_date,
        location_id,
        avg_discount,
        False  # Default assumption that promotions are for everyone
    ))

# Step 7: Commit and close
conn.commit()
cur.close()
conn.close()

print("Promotions inserted successfully!")


Promotions inserted successfully!


For promotions, since there were already discount rates applied and the discount_percentage has a NOT NULL constraint, we computed the average for each type of deal.

In [403]:
# 13) Sales orders

sales_customers_clean = pd.read_excel('/Users/kimminsung/Desktop/Columbia/SQL/SQL Final Project/ABC_sales_and_customers.xlsx')

conn = psycopg2.connect(
    dbname='sql_final_project',
    user='postgres',
    password='123',
    host='localhost',
    port='5432'
)
cur = conn.cursor()

for index, row in sales_customers_clean.iterrows():
    # Lookup location_id
    cur.execute("SELECT location_id FROM locations WHERE location_name = %s", (row['location_name'],))
    location = cur.fetchone()
    if not location:
        print(f"Location '{row['location_name']}' not found, skipping.")
        continue
    location_id = location[0]

    # SKIP if no customer names
    if pd.isna(row['customer_first_name']) or pd.isna(row['customer_last_name']):
        customer_id = None
    else:
        # Lookup customer_id normally
        cur.execute("""
            SELECT customer_id
            FROM customers
            WHERE first_name = %s AND last_name = %s
            LIMIT 1
        """, (row['customer_first_name'], row['customer_last_name']))
        customer = cur.fetchone()
        customer_id = customer[0] if customer else None

    # Insert into sales_orders
    cur.execute("""
        INSERT INTO sales_orders (order_date, location_id, customer_id)
        VALUES (%s, %s, %s)
    """, (
        row['sale_date'],
        location_id,
        customer_id
    ))

conn.commit()
cur.close()
conn.close()

print("Sales orders inserted successfully!")

Sales orders inserted successfully!


In [410]:
# 14) Sales order details

sales_customers_clean = pd.read_csv('/Users/kimminsung/Desktop/Columbia/SQL/SQL Final Project/ABC_sales_and_customers.csv')

conn = psycopg2.connect(
    dbname='sql_final_project',
    user='postgres',
    password='123',
    host='localhost',
    port='5432'
)
cur = conn.cursor()

# Loop through the sales_customers_clean and insert into sale_order_details
for index, row in sales_customers_clean.iterrows():
    # Find the sale_order_id (matching by order_date and location)
    cur.execute("""
        SELECT sale_order_id 
        FROM sales_orders 
        WHERE order_date = %s 
        AND location_id = (
            SELECT location_id FROM locations WHERE location_name = %s
        )
        LIMIT 1
    """, (row['sale_date'], row['location_name']))
    sale_order_result = cur.fetchone()
    
    if sale_order_result is None:
        print(f"Warning: Sale order not found for date {row['sale_date']} and location {row['location_name']}. Skipping.")
        continue
    sale_order_id = sale_order_result[0]

    # Find the product_id
    cur.execute("""
        SELECT product_id 
        FROM products 
        WHERE product_name = %s
        LIMIT 1
    """, (row['product_name'],))
    product_result = cur.fetchone()
    
    if product_result is None:
        print(f"Warning: Product '{row['product_name']}' not found. Skipping.")
        continue
    product_id = product_result[0]

    # Find the promotion_id (if promotion name is not null)
    promotion_id = None
    if pd.notna(row['promotion_name']):
        cur.execute("""
            SELECT promotion_id 
            FROM promotions 
            WHERE promotion_name = %s
            LIMIT 1
        """, (row['promotion_name'],))
        promo_result = cur.fetchone()
        if promo_result:
            promotion_id = promo_result[0]
        else:
            print(f"Warning: Promotion '{row['promotion_name']}' not found. Setting promotion_id = NULL.")

    # Step 4: Insert into sale_order_details
    cur.execute("""
        INSERT INTO sale_order_details (sale_order_id, product_id, quantity, unit_price, discount, promotion_id)
        VALUES (%s, %s, %s, %s, %s, %s)
    """, (
        sale_order_id,
        product_id,
        row['quantity'],
        row['unit_price'],
        row['discount_applied'],
        promotion_id
    ))

conn.commit()
cur.close()
conn.close()

print("Sale order details inserted successfully!")

Sale order details inserted successfully!


In [412]:
# 15) Transactions (populate table from sales and purchase orders)
import psycopg2

conn = psycopg2.connect(
    dbname='sql_final_project',
    user='postgres',
    password='123',
    host='localhost',
    port='5432'
)
cur = conn.cursor()

# Insert Sales Transactions
cur.execute("""
    INSERT INTO transactions (transaction_date, amount, transaction_type, payment_type, sale_order_id, purchase_order_id)
    SELECT 
        order_date,
        total_amount,
        'sale' AS transaction_type,
        NULL AS payment_type,
        sale_order_id,
        NULL AS purchase_order_id
    FROM sales_orders;
""")
print("Sales transactions inserted.")

# Insert Purchase Transactions
cur.execute("""
    INSERT INTO transactions (transaction_date, amount, transaction_type, payment_type, sale_order_id, purchase_order_id)
    SELECT 
        po.order_date,
        SUM(pod.quantity * pod.unit_cost) AS amount,
        'purchase' AS transaction_type,
        NULL AS payment_type,
        NULL AS sale_order_id,
        po.purchase_order_id
    FROM purchase_orders po
    JOIN purchase_order_details pod
    ON po.purchase_order_id = pod.purchase_order_id
    WHERE po.status = 'Received'
    GROUP BY po.purchase_order_id, po.order_date;
""")
print("Purchase transactions inserted.")

# Commit and Close
conn.commit()
cur.close()
conn.close()

print("Transactions inserted successfully!")

Sales transactions inserted.
Purchase transactions inserted.
Transactions inserted successfully!


In [439]:
# 16 Expenses (Will take the purchase orders to get procurement, then load data from accounting.csv)
# Must locate the location_id since the file contains the name of the location where the expense ocurred.

conn = psycopg2.connect(
    dbname='sql_final_project',
    user='postgres',
    password='123',
    host='localhost',
    port='5432'
)
cur = conn.cursor()

accounting_expenses = pd.read_csv('/Users/kimminsung/Desktop/Columbia/SQL/SQL Final Project/ABC_accounting.csv')

# Lookup dictionaries
# Locations
cur.execute("SELECT location_id, location_name FROM locations;")
location_lookup = {name: loc_id for loc_id, name in cur.fetchall()}

# Suppliers
cur.execute("SELECT supplier_id, supplier_name FROM suppliers;")
supplier_lookup = {name: sid for sid, name in cur.fetchall()}

# Products
cur.execute("SELECT product_id, product_name FROM products;")
product_lookup = {name: pid for pid, name in cur.fetchall()}

# Insert Procurement Expenses (real quantities and unit_costs)
# Query to join purchase_orders + purchase_order_details
cur.execute("""
    SELECT po.purchase_order_id, po.order_date, po.location_id, po.supplier_id,
           pod.product_id, pod.quantity, pod.unit_cost
    FROM purchase_orders po
    JOIN purchase_order_details pod ON po.purchase_order_id = pod.purchase_order_id
    WHERE po.status = 'Received';
""")
procurement_rows = cur.fetchall()

for row in procurement_rows:
    purchase_order_id, order_date, location_id, supplier_id, product_id, quantity, unit_cost = row
    amount = round(quantity * unit_cost, 2)

    cur.execute("""
        INSERT INTO expenses (category, amount, expense_date, location_id, supplier_id, description)
        VALUES (%s, %s, %s, %s, %s, %s);
    """, (
        'Procurement',
        amount,
        order_date,
        location_id,
        supplier_id,
        f'Procurement of product_id {product_id} (qty: {quantity})'
    ))

print("Procurement expenses inserted.")

# Insert Operating Expenses (accounting CSV)

for index, row in accounting_expenses.iterrows():
    location_id = location_lookup.get(row['location_name'])
    if location_id is None:
        print(f"Warning: Location '{row['location_name']}' not found, skipping row.")
        continue

    cur.execute("""
        INSERT INTO expenses (category, amount, expense_date, location_id, supplier_id, description)
        VALUES (%s, %s, %s, %s, NULL, %s);
    """, (
        row['category'],
        row['amount'],
        row['expense_date'],
        location_id,
        row['description']
    ))

print("Operating expenses (ABC_accounting.csv) inserted.")

conn.commit()
cur.close()
conn.close()

print("Expenses inserted successfully!")

Procurement expenses inserted.
Operating expenses (ABC_accounting.csv) inserted.
Expenses inserted successfully!


In [446]:
# 17) Returns
# Since the client didn't really document returns, we propose to make a different return table.
# This will help them track lost revenue and hidden costs.
# Fortunately, they documented some transactions in the sales and customers table where they added 'returned' in the notes.
# Therefore, we will only load the sales id, s.
# We will leave all the other as null values.

conn = psycopg2.connect(
    dbname='sql_final_project',
    user='postgres',
    password='123',
    host='localhost',
    port='5432'
)
cur = conn.cursor()

# Load sales and customers data
sales_customers = pd.read_csv('/Users/kimminsung/Desktop/Columbia/SQL/SQL Final Project/ABC_sales_and_customers.csv')

# Step 1: Filter only returned sales
returned_sales = sales_customers[sales_customers['notes'] == 'Returned']

print(f"Found {len(returned_sales)} returned sales to insert into returns table.")

# Loop through and insert into returns table
for index, row in returned_sales.iterrows():
    # First lookup sale_order_detail_id
    cur.execute("""
        SELECT sod.sale_order_detail_id
        FROM sales_orders so
        JOIN sale_order_details sod ON so.sale_order_id = sod.sale_order_id
        JOIN products p ON sod.product_id = p.product_id
        WHERE so.order_date = %s
          AND p.product_name = %s
          AND sod.quantity = %s
        LIMIT 1;
    """, (
        row['sale_date'],
        row['product_name'],
        row['quantity']
    ))
    result = cur.fetchone()

    if result is None:
        print(f"Warning: Could not find matching sale order for product {row['product_name']} on {row['sale_date']}. Skipping...")
        continue

    sale_order_detail_id = result[0]

    # Insert into returns table
    cur.execute("""
        INSERT INTO returns (order_detail_id, quantity_returned, return_date, reason)
        VALUES (%s, %s, NULL, NULL)
    """, (
        sale_order_detail_id,
        row['quantity']
    ))

conn.commit()
cur.close()
conn.close()

print("Returns inserted successfully!")

Found 30 returned sales to insert into returns table.
Returns inserted successfully!


This concludes the succesful data migration from the client to the newly formed schema.